# Extract GitHub Copilot Metrics for users and enterprise

In [ ]:
# Import necessary libraries
import json

import pandas as pd
import requests

In [ ]:
# Load .env variables
from dotenv import load_dotenv
import os
load_dotenv()

In [ ]:
# Define helpers to download, parse, and save metrics payloads.

def parse_response_json(response):
    try:
        return response.json()
    except ValueError:
        text = response.text.strip()
        if not text:
            return None

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            # Some report files are newline-delimited JSON (JSONL/NDJSON).
            lines = [line for line in text.splitlines() if line.strip()]
            try:
                return [json.loads(line) for line in lines]
            except json.JSONDecodeError as exc:
                preview = text[:300].replace("\n", " ")
                print(f"JSON parsing failed: {exc}")
                print(f"Response preview: {preview}...")
                return None


def resolve_download_url(link):
    if isinstance(link, dict):
        return link.get("url")
    if isinstance(link, str):
        return link
    return None


def download_json(url, request_headers=None):
    response = requests.get(url, headers=request_headers)
    if response.status_code == 200:
        return parse_response_json(response)

    print(f"Error: {response.status_code} - {response.text}")
    return None


def write_json_to_path(payload, file_path):
    directory = os.path.dirname(file_path)
    if directory:
        os.makedirs(directory, exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as fp:
        json.dump(payload, fp, indent=2)

    print(f"Saved JSON to: {file_path}")

In [ ]:
enterprise_28_days_url = f" https://api.github.com/enterprises/{os.getenv('ENTERPRISE_NAME')}/copilot/metrics/reports/enterprise-28-day/latest"
users_28_days_url = f"https://api.github.com/enterprises/{os.getenv('ENTERPRISE_NAME')}/copilot/metrics/reports/users-28-day/latest"
headers = {
    "Authorization": f"Bearer {os.getenv('BEARER_TOKEN')}"
}
print(enterprise_28_days_url)
print(users_28_days_url)


In [ ]:
response = requests.get(enterprise_28_days_url, headers=headers)
if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Error: {response.status_code} - {response.text}")

# download_links can be either URL strings or objects containing a "url" field
json_link = data.get("download_links")
if json_link:
    for index, link in enumerate(json_link, start=1):
        download_url = resolve_download_url(link)
        if not download_url:
            print(f"Skipping invalid download link entry: {link}")
            continue

        print(f"Downloading from {download_url}...")
        request_headers = headers if "api.github.com" in download_url else None
        json_data = download_json(download_url, request_headers=request_headers)
        if json_data is not None:
            output_path = f"data/enterprise_28_day_report_{index}.json"
            write_json_to_path(json_data, output_path)
else:
    print("No download links found in the response.")

In [ ]:
response = requests.get(users_28_days_url, headers=headers)
if response.status_code == 200:
    data = response.json()
    print(data)
else:
    print(f"Error: {response.status_code} - {response.text}")

# download_links can be either URL strings or objects containing a "url" field
json_link = data.get("download_links")
if json_link:
    for index, link in enumerate(json_link, start=1):
        download_url = resolve_download_url(link)
        if not download_url:
            print(f"Skipping invalid download link entry: {link}")
            continue

        print(f"Downloading from {download_url}...")
        request_headers = headers if "api.github.com" in download_url else None
        json_data = download_json(download_url, request_headers=request_headers)
        if json_data is not None:
            output_path = f"data/users_28_day_report_{index}.json"
            write_json_to_path(json_data, output_path)
else:
    print("No download links found in the response.")

In [ ]:
# Create a chart showing active users by day for the last 28 days using users-28-day report data.
import matplotlib.pyplot as plt

users_report_path = "data/users_28_day_report_1.json"

with open(users_report_path, "r", encoding="utf-8") as fp:
    users_data = json.load(fp)

users_df = pd.DataFrame(users_data)
users_df["day"] = pd.to_datetime(users_df["day"])

activity_cols = [
    "user_initiated_interaction_count",
    "code_generation_activity_count",
    "code_acceptance_activity_count",
]

# Treat a user-day as active if any core activity metric is greater than zero.
users_df["is_active"] = users_df[activity_cols].fillna(0).sum(axis=1) > 0

active_users_by_day = (
    users_df.loc[users_df["is_active"]]
    .groupby("day", as_index=False)["user_id"]
    .nunique()
    .rename(columns={"user_id": "active_users"})
    .sort_values("day")
)

# Keep only the most recent 28 days available in the report.
active_users_by_day = active_users_by_day.tail(28)

plt.figure(figsize=(12, 5))
plt.plot(active_users_by_day["day"], active_users_by_day["active_users"], marker="o")
plt.title("Active Copilot Users by Day (Last 28 Days)")
plt.xlabel("Day")
plt.ylabel("Active Users")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

active_users_by_day

In [ ]:
# Show the most active users by code generated, prompts sent, and prompts accepted.
metrics = [
    "code_generation_activity_count",
    "user_initiated_interaction_count",
    "code_acceptance_activity_count",
]

user_activity = (
    users_df.groupby(["user_id", "user_login"], as_index=False)[metrics]
    .sum()
    .rename(
        columns={
            "code_generation_activity_count": "code_generated",
            "user_initiated_interaction_count": "prompts_sent",
            "code_acceptance_activity_count": "prompts_accepted",
        }
    )
)

user_activity["total_activity"] = (
    user_activity["code_generated"]
    + user_activity["prompts_sent"]
    + user_activity["prompts_accepted"]
)

top_n = 10

top_by_code_generated = user_activity.sort_values("code_generated", ascending=False).head(top_n)
top_by_prompts_sent = user_activity.sort_values("prompts_sent", ascending=False).head(top_n)
top_by_prompts_accepted = user_activity.sort_values("prompts_accepted", ascending=False).head(top_n)
top_overall = user_activity.sort_values("total_activity", ascending=False).head(top_n)

print("Top users by code generated")
display(top_by_code_generated[["user_login", "code_generated", "prompts_sent", "prompts_accepted", "total_activity"]])

print("Top users by prompts sent")
display(top_by_prompts_sent[["user_login", "prompts_sent", "code_generated", "prompts_accepted", "total_activity"]])

print("Top users by prompts accepted")
display(top_by_prompts_accepted[["user_login", "prompts_accepted", "code_generated", "prompts_sent", "total_activity"]])

print("Top users overall (combined activity)")
display(top_overall[["user_login", "total_activity", "code_generated", "prompts_sent", "prompts_accepted"]])